In [1]:
import pennylane as QuantumIoT
import numpy as np
import matplotlib.pyplot as plt

from pennylane import qaoa

In [2]:
coeffs = [
    -0.75,
    -1.25,
    -2.0,
     2.75
]

ops = [
    QuantumIoT.Z(0),
    QuantumIoT.Z(1),
    QuantumIoT.Z(2),
    QuantumIoT.Z(0) @ QuantumIoT.Z(1)
]

cost_h = QuantumIoT.Hamiltonian(coeffs, ops)

In [3]:
# ============================================
# Mixer Hamiltonian
# ============================================

mixer_h = qaoa.x_mixer(wires=[0,1,2])

In [4]:
print("The Cost Hamiltonian is:")
print(cost_h)
print("\n")
print("And the Mixer Hamiltonian is:")
print(mixer_h)

The Cost Hamiltonian is:
-0.75 * Z(0) + -1.25 * Z(1) + -2.0 * Z(2) + 2.75 * (Z(0) @ Z(1))


And the Mixer Hamiltonian is:
1 * X(0) + 1 * X(1) + 1 * X(2)


In [5]:
dev = QuantumIoT.device("default.qubit", wires=3)

In [6]:
@QuantumIoT.qnode(dev)
def QAOASensorEnergy(gamma, beta):

    for w in range(3):
        QuantumIoT.Hadamard(wires=w)

    qaoa.cost_layer(gamma, cost_h)

    qaoa.mixer_layer(beta, mixer_h)

    return QuantumIoT.expval(cost_h)

In [7]:
params = QuantumIoT.numpy.random.normal(size=2)
opt = QuantumIoT.AdamOptimizer(stepsize=0.1)

In [8]:
for step in range(100):

    params = opt.step(
        lambda p: QAOASensorEnergy(p[0], p[1]),
        params
    )

    if (step + 1) % 10 == 0:
        energy = QAOASensorEnergy(params[0], params[1])
        print(f"Step {step+1:3d} | Total Sensor Energy = {energy:.6f}")          

Step  10 | Total Sensor Energy = -3.830481
Step  20 | Total Sensor Energy = -3.833584
Step  30 | Total Sensor Energy = -3.834333
Step  40 | Total Sensor Energy = -3.838979
Step  50 | Total Sensor Energy = -3.840706
Step  60 | Total Sensor Energy = -3.842082
Step  70 | Total Sensor Energy = -3.842301
Step  80 | Total Sensor Energy = -3.842400
Step  90 | Total Sensor Energy = -3.842440
Step 100 | Total Sensor Energy = -3.842471


In [9]:
# ============================================
# Final Results
# ============================================

final_energy = QAOASensorEnergy(params[0], params[1])

print("\nOptimized Parameters:")
print(params)

print("\nGround-State Energy:")
print(final_energy)


Optimized Parameters:
[-1.8501245   0.61383405]

Ground-State Energy:
-3.842470676589948


In [10]:
@QuantumIoT.qnode(dev, shots=2048)
def QAOASamplingSensor(gamma, beta):

    for w in range(3):
        QuantumIoT.Hadamard(wires=w)

    qaoa.cost_layer(gamma, cost_h)

    qaoa.mixer_layer(beta, mixer_h)

    return QuantumIoT.sample()

In [11]:
samples = QAOASamplingSensor(params[0], params[1])

In [12]:
print("\n================ SAMPLES FROM QAOA Sensor ENERGY CIRCUIT ================\n")
print(samples)


================ SAMPLES FROM QAOA Sensor ENERGY CIRCUIT ================

[[0 0 0]
 [1 0 0]
 [1 0 0]
 ...
 [1 0 1]
 [0 0 0]
 [1 0 0]]


In [13]:
most_from_samples = np.array([tuple(sample) for sample in samples])
(unique, counts) = np.unique(most_from_samples, axis=0, return_counts=True)
print("\nUnique bitstrings and their counts:")
for bitstring, count in zip(unique, counts):
    print(f"{bitstring} -> {count} times")


Unique bitstrings and their counts:
[0 0 0] -> 404 times
[0 0 1] -> 37 times
[0 1 0] -> 285 times
[0 1 1] -> 22 times
[1 0 0] -> 1199 times
[1 0 1] -> 88 times
[1 1 0] -> 12 times
[1 1 1] -> 1 times


In [14]:
from platform import python_version

print('Pennylane Version:', QuantumIoT.__version__)

print('Python Version:', python_version())
print('UdeA-Medellin (2026) ©')

Pennylane Version: 0.43.1
Python Version: 3.11.7
UdeA-Medellin (2026) ©
